# 03 – DQ Runner (schedule this)
For each active rule: count the population (`base_sql`), find failing keys (`failure_condition` on the base,
or `failure_sql`), write history + run log. Then rebuild the current worklist and close the run.
One failing rule does not stop the others.

In [ ]:
%run 00_dq_config

In [ ]:
# Step 1: start the run
run_ts   = now_local()
run_date = run_ts.date()
run_id   = str(uuid.uuid4())

if REPLACE_SAME_DAY_RUN:
    for tbl in [T_RUN_LOG, T_FAILED_HIST, T_RUN]:
        spark.sql(f"DELETE FROM {tbl} WHERE run_date = DATE'{run_date}'")

spark.sql(f"""
INSERT INTO {T_RUN} VALUES
('{run_id}', DATE'{run_date}', TIMESTAMP'{run_ts}', NULL, 'RUNNING', NULL, NULL)
""")
print(f"Run {run_id} started for {run_date}")

In [ ]:
# Step 2: execute rules
rules = (spark.table(T_CATALOG)
              .filter("is_active = true")
              .orderBy("rule_id")
              .collect())

log_rows = []

for r in rules:
    t0 = time.time()
    print(f"Rule {r.rule_id}: {r.rule_name}")
    base = None
    try:
        # population (denominator)
        base  = spark.sql(r.base_sql).cache()
        total = base.select("record_key").distinct().count()

        # failing records
        if r.failure_condition:
            failed_src = base.filter(r.failure_condition)
        else:
            failed_src = spark.sql(r.failure_sql)

        (failed_src.select(F.col("record_key").cast("string").alias("record_key"))
                   .distinct()
                   .select(F.lit(run_id).alias("run_id"),
                           F.lit(run_date).cast("date").alias("run_date"),
                           F.lit(r.rule_id).cast("int").alias("rule_id"),
                           "record_key")
                   .write.mode("append").format("delta").saveAsTable(T_FAILED_HIST))

        failed = (spark.table(T_FAILED_HIST)
                       .filter((F.col("run_date") == F.lit(run_date)) &
                               (F.col("run_id") == run_id) &
                               (F.col("rule_id") == r.rule_id))
                       .count())

        passed    = total - failed
        pass_rate = round(passed / total, 6) if total > 0 else None
        log_rows.append((run_id, run_date, run_ts, r.rule_id, r.rule_version,
                         total, failed, passed, pass_rate, "SUCCESS", None,
                         round(time.time() - t0, 1)))
        print(f"   total={total}  failed={failed}  pass_rate={pass_rate}")

    except Exception as e:
        log_rows.append((run_id, run_date, run_ts, r.rule_id, r.rule_version,
                         None, None, None, None, "FAILED", str(e)[:2000],
                         round(time.time() - t0, 1)))
        print(f"   FAILED: {str(e)[:300]}")
    finally:
        if base is not None:
            base.unpersist()

In [ ]:
# Step 3: write the run log
if log_rows:
    (spark.createDataFrame(log_rows, spark.table(T_RUN_LOG).schema)
          .write.mode("append").format("delta").saveAsTable(T_RUN_LOG))

In [ ]:
# Step 4: rebuild the current worklist (failures of this run + how long each has been failing)
first_seen = (spark.table(T_FAILED_HIST)
                   .groupBy("rule_id", "record_key")
                   .agg(F.min("run_date").alias("first_failed_date")))

current = (spark.table(T_FAILED_HIST)
                .filter(F.col("run_id") == run_id)
                .join(first_seen, ["rule_id", "record_key"], "left")
                .withColumn("days_failing",
                            (F.datediff("run_date", "first_failed_date") + 1).cast("int"))
                .select("rule_id", "record_key", "run_id", "run_date",
                        "first_failed_date", "days_failing"))

current.write.mode("overwrite").format("delta").saveAsTable(T_FAILED_CURR)

In [ ]:
# Step 5: history retention
spark.sql(f"""
DELETE FROM {T_FAILED_HIST}
WHERE run_date < date_sub(DATE'{run_date}', {HISTORY_RETENTION_DAYS})
""")

In [ ]:
# Step 6: close the run
n_exec  = len(log_rows)
n_error = sum(1 for x in log_rows if x[9] == "FAILED")
status  = "SUCCESS" if n_error == 0 else ("FAILED" if n_error == n_exec else "PARTIAL")

spark.sql(f"""
UPDATE {T_RUN}
SET end_ts = TIMESTAMP'{now_local()}',
    status = '{status}',
    rules_executed = {n_exec},
    rules_errored  = {n_error}
WHERE run_id = '{run_id}'
""")

print(f"Run {run_id} finished: {status} ({n_exec} rules, {n_error} errored)")
display(spark.table(T_RUN_LOG).filter(F.col("run_id") == run_id).orderBy("rule_id"))

In [ ]:
# Step 7: return status to the pipeline (shows as the notebook activity's exit value)
notebookutils.notebook.exit(status)